# JobShield – 03 Baseline Model (Week 3)

**Group 5 · AML-3403 AI and ML Capstone · Week 3 lead: Diksha Sharma**
Repository: https://github.com/Krushal16/JobShield

1. Train the baselines – TF-IDF + Logistic Regression (main) and TF-IDF + Naive Bayes (comparison) *(Diksha)*
2. Confusion matrix, precision-recall curves and threshold trade-off *(Diksha)*
3. TF-IDF tuning with 5-fold cross-validation on the training set *(Arti)*
4. Words the model relies on *(Narinder)*
5. Error analysis – missed scams and false alarms *(Narinder)*
6. Save the model and score single postings with `predict_posting()` *(Krushal)*
7. Summary for Weekly Report 3

**Needs:** `data/processed/train.csv` and `val.csv` from notebook 02. The **test set is not used** until Week 7.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from src.data.loader import TARGET
from src.data.split import load_splits
from src.training.train_baseline import train_and_evaluate, save_model, top_features
from src.training.tune_tfidf import tune_tfidf
from src.models.baseline import build_logreg_pipeline, fit_pipeline
from src.evaluation.metrics import evaluate, print_report, log_result, plot_confusion, plot_pr_curve
from src.evaluation.error_analysis import error_table, threshold_table, missed_scam_profile, plot_top_features
from src.inference.predict import predict_posting, load_model
from src.utils.config import load_config, project_path, set_seed

CFG = load_config(); set_seed(CFG["seed"])
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
train_df, val_df, _ = load_splits()
print(f"train {len(train_df):,} rows ({train_df[TARGET].sum()} fake) | val {len(val_df):,} rows ({val_df[TARGET].sum()} fake)")
summary = {"train_rows": len(train_df), "val_rows": len(val_df), "val_fake": int(val_df[TARGET].sum())}

## 1. Train the baselines
Each model is scored twice: at the default threshold 0.5 and at the threshold that maximises F1 on validation.

In [ ]:
out = train_and_evaluate(train_df, val_df, CFG, log=True)
res = pd.DataFrame(out["results"])[["model", "notes", "precision", "recall", "f1", "pr_auc", "roc_auc", "tp", "fn", "fp"]]
res.round(3)

## 2. Confusion matrix, precision-recall curves and threshold trade-off

In [ ]:
LR, NB = "TF-IDF + Logistic Regression", "TF-IDF + Naive Bayes"
y_val = val_df[TARGET].values
p_lr, p_nb = out["probs"][LR], out["probs"][NB]
thr_lr = out["thresholds"][LR]

lr_tuned = [r for r in out["results"] if r["model"] == LR and "F1-best" in r["notes"]][0]
plot_confusion(lr_tuned, save_as="09_baseline_confusion.png"); plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
plot_pr_curve(y_val, p_lr, label="Logistic Regression", ax=ax)
plot_pr_curve(y_val, p_nb, label="Naive Bayes", ax=ax, save_as="10_baseline_pr_curves.png"); plt.show()

print("Threshold trade-off – Logistic Regression (validation):")
threshold_table(y_val, p_lr)

## 3. TF-IDF tuning (5-fold CV on the training set only)
Takes a few minutes on Colab. If the best setting differs from `configs/config.yaml`, update the config and re-run section 1.

In [ ]:
tuning = tune_tfidf(train_df, CFG)
best = tuning.iloc[0]
summary["tfidf_best"] = f"ngram={best.ngram_range}, min_df={best.min_df}, max_features={best.max_features}, CV PR-AUC={best.pr_auc_mean:.3f}±{best.pr_auc_std:.3f}"
tuning.head(6)

## 4. Words the model relies on

In [ ]:
pipe_lr = out["models"][LR]
top = top_features(pipe_lr, n=20)
plot_top_features(top, n=15, save_as="11_baseline_top_words.png"); plt.show()
summary["top_fake_words"] = top[top.direction == "fake"].word.head(10).tolist()
summary["top_real_words"] = top[top.direction == "real"].word.head(10).tolist()

## 5. Error analysis
Which fake postings does the model miss, and which real postings does it wrongly flag? These findings feed the Week 4 red-flag features.

In [ ]:
fn = error_table(val_df, p_lr, threshold=thr_lr, kind="fn")
fp = error_table(val_df, p_lr, threshold=thr_lr, kind="fp")
summary["missed_scams"] = int(((y_val == 1) & (p_lr < thr_lr)).sum())
summary["false_alarms"] = int(((y_val == 0) & (p_lr >= thr_lr)).sum())
print(f"Missed scams: {summary['missed_scams']} | False alarms: {summary['false_alarms']}")
print("\nCaught vs missed fake postings (share with each signal):")
print(missed_scam_profile(val_df, p_lr, threshold=thr_lr).to_string())
fn

In [ ]:
fp

## 6. Save the model and score single postings
The saved file goes to `models/final/` (not committed to Git).

In [ ]:
path = save_model(pipe_lr, thr_lr, cfg=CFG)
print("Saved:", path.relative_to(ROOT))
bundle = load_model(path)

examples = {
    "Suspicious (illustrative)": dict(title="Data Entry Assistant – Work From Home",
        description="Earn $500 weekly from home. No experience needed. Contact us on WhatsApp to start immediately. "
                    "A cheque for equipment will be sent to you."),
    "Typical real (illustrative)": dict(title="Junior Data Analyst",
        company_profile="We are a Toronto-based analytics consultancy serving retail clients since 2009.",
        description="You will build dashboards in Power BI, clean data with SQL and Python, and present insights to clients.",
        requirements="Degree in statistics or computer science; 1+ year of SQL experience.",
        benefits="Health and dental plan, hybrid work, training budget."),
}
for name, ex in examples.items():
    r = predict_posting(**ex, model_bundle=bundle)
    print(f"{name}: P(fake)={r['probability_fake']:.2f} -> {r['label']} ({r['risk_level']} risk)")
    print("   towards fake:", r["words_towards_fake"])
    print("   towards real:", r["words_towards_real"])

## 7. Summary for Weekly Report 3

In [ ]:
for r in out["results"]:
    key = ("lr" if "Logistic" in r["model"] else "nb") + ("_tuned" if "F1-best" in r["notes"] else "_0.5")
    summary[key] = f"P={r['precision']:.3f} R={r['recall']:.3f} F1={r['f1']:.3f} PR-AUC={r['pr_auc']:.3f}"
summary["lr_threshold"] = round(thr_lr, 3)
for k, v in summary.items():
    print(f"{k:16s}: {v}")